<a href="https://colab.research.google.com/github/asiahcrutchfield/ground-level-languages-ai/blob/main/gll_intro_to_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# Ground Level Languages — Update #1 Acoustic Alignment & Pitch Pipeline
# Models: Allosaurus (Phoneme Detection -> Clean Reference Bank) + TorchCREPE
# ==============================================================================

!pip install -q allosaurus torchcrepe librosa soundfile scipy numpy

import os
import json
import zipfile
import urllib.request
import numpy as np
import torch
import torchcrepe
import librosa
import soundfile as sf
from scipy.signal import find_peaks
from allosaurus.app import read_recognizer

# 1. Setup output directories
os.makedirs("audio/slices", exist_ok=True)
os.makedirs("audio/phonemes", exist_ok=True)

# 2. Download clean, studio-recorded isolated IPA audio from Wikimedia Commons
WIKIMEDIA_IPA_URLS = {
    "k":  ("k.wav",  "https://upload.wikimedia.org/wikipedia/commons/e/e3/Voiceless_velar_plosive.ogg"),
    "kʰ": ("k.wav",  "https://upload.wikimedia.org/wikipedia/commons/e/e3/Voiceless_velar_plosive.ogg"),
    "t":  ("t.wav",  "https://upload.wikimedia.org/wikipedia/commons/0/02/Voiceless_alveolar_plosive.ogg"),
    "tʰ": ("t.wav",  "https://upload.wikimedia.org/wikipedia/commons/0/02/Voiceless_alveolar_plosive.ogg"),
    "d":  ("d.wav",  "https://upload.wikimedia.org/wikipedia/commons/0/01/Voiced_alveolar_plosive.ogg"),
    "ɡ":  ("g.wav",  "https://upload.wikimedia.org/wikipedia/commons/1/12/Voiced_velar_plosive_02.ogg"),
    "g":  ("g.wav",  "https://upload.wikimedia.org/wikipedia/commons/1/12/Voiced_velar_plosive_02.ogg"),
    "w":  ("w.wav",  "https://upload.wikimedia.org/wikipedia/commons/f/f2/Voiced_labio-velar_approximant.ogg"),
    "l":  ("l.wav",  "https://upload.wikimedia.org/wikipedia/commons/b/bc/Alveolar_lateral_approximant.ogg"),
    "r":  ("r.wav",  "https://upload.wikimedia.org/wikipedia/commons/1/1f/Alveolar_approximant.ogg"),
    "ɹ":  ("r.wav",  "https://upload.wikimedia.org/wikipedia/commons/1/1f/Alveolar_approximant.ogg"),
    "dʒ": ("dz.wav", "https://upload.wikimedia.org/wikipedia/commons/e/e6/Voiced_palato-alveolar_affricate.ogg"),
    "m":  ("m.wav",  "https://upload.wikimedia.org/wikipedia/commons/a/a9/Bilabial_nasal.ogg"),
    "p":  ("p.wav",  "https://upload.wikimedia.org/wikipedia/commons/5/51/Voiceless_bilabial_plosive.ogg"),
    "æ":  ("ae.wav", "https://upload.wikimedia.org/wikipedia/commons/c/c9/Near-open_front_unrounded_vowel.ogg"),
    "ɑ":  ("aa.wav", "https://upload.wikimedia.org/wikipedia/commons/e/e5/Open_back_unrounded_vowel.ogg"),
    "ɔ":  ("ao.wav", "https://upload.wikimedia.org/wikipedia/commons/d/d0/PR-open-mid_back_rounded_vowel.ogg"),
    "ʌ":  ("ah.wav", "https://upload.wikimedia.org/wikipedia/commons/9/92/Open-mid_back_unrounded_vowel.ogg"),
    "i":  ("iy.wav", "https://upload.wikimedia.org/wikipedia/commons/9/91/Close_front_unrounded_vowel.ogg"),
    "iː": ("iy.wav", "https://upload.wikimedia.org/wikipedia/commons/9/91/Close_front_unrounded_vowel.ogg"),
}

def apply_fade(audio_clip, sr=16000, fade_ms=15):
    """Applies a smooth fade-in and fade-out to prevent harsh audio clicks."""
    fade_len = min(int(sr * (fade_ms / 1000.0)), len(audio_clip) // 2)
    if fade_len <= 1:
        return audio_clip
    window = np.ones(len(audio_clip), dtype=np.float32)
    window[:fade_len] = np.linspace(0.0, 1.0, fade_len)
    window[-fade_len:] = np.linspace(1.0, 0.0, fade_len)
    return audio_clip * window

print("Downloading isolated Wikimedia Commons IPA phoneme recordings...")
headers = {"User-Agent": "GroundLevelLanguagesResearch/1.0"}
for ipa_key, (fname, url) in WIKIMEDIA_IPA_URLS.items():
    out_path = f"audio/phonemes/{fname}"
    if not os.path.exists(out_path):
        temp_ogg = f"/content/temp_{fname}.ogg"
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req) as resp, open(temp_ogg, "wb") as out_f:
                out_f.write(resp.read())
            clip, _ = librosa.load(temp_ogg, sr=16000)
            clip, _ = librosa.effects.trim(clip, top_db=30)
            clip = apply_fade(clip, sr=16000, fade_ms=15)
            sf.write(out_path, clip, 16000)
        except Exception as e:
            print(f"Warning: Could not download {fname} ({url}): {e}")

# 3. Load Allosaurus & TorchCREPE models
recognizer = read_recognizer()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Models loaded on device: {device}\n")

IPA_VOWELS = set("aeiouyɨʉɯuɪʏʊeøɘɵɤoəɛœɜɞʌɔæɐaɶɑɒɚɝ")

def classify_phone(ipa_symbol):
    return "vowel" if any(ch in IPA_VOWELS for ch in ipa_symbol) else "consonant"

def synthesize_humming_track(y, f0, periodicity, sr=16000, hop_length=160):
    """Generates a pure vocal humming track gated by speech energy + CREPE F0 contours."""
    total_samples = len(y)
    freq_interp = np.interp(np.arange(total_samples), np.linspace(0, total_samples, len(f0)), f0)
    conf_interp = np.interp(np.arange(total_samples), np.linspace(0, total_samples, len(periodicity)), periodicity)

    rms = librosa.feature.rms(y=y, frame_length=512, hop_length=hop_length)[0]
    rms_interp = np.interp(np.arange(total_samples), np.linspace(0, total_samples, len(rms)), rms)
    rms_norm = rms_interp / (np.max(rms_interp) + 1e-6)

    voiced_mask = (conf_interp > 0.15) & (rms_norm > 0.12)
    amp = np.where(voiced_mask, 0.35 * rms_norm, 0.0)

    phase = np.divide(np.multiply(2.0 * np.pi, np.cumsum(freq_interp)), sr)
    fundamental = np.multiply(0.8, np.sin(phase))
    harmonic = np.multiply(0.2, np.sin(np.multiply(2.0, phase)))
    return np.multiply(amp, fundamental + harmonic).astype(np.float32)

def process_word(word_id, mp3_path, visual_path):
    y_raw, sr = librosa.load(mp3_path, sr=16000)
    y, _ = librosa.effects.trim(y_raw, top_db=28)
    y = np.pad(y, (int(0.04 * sr), int(0.04 * sr)), mode="constant")

    temp_wav = f"/content/{word_id}_16k.wav"
    sf.write(temp_wav, y, sr)

    # --------------------------------------------------------------------------
    # STEP 1: Allosaurus Phoneme Detection -> Clean Wikimedia Reference Mapping
    # --------------------------------------------------------------------------
    raw_phones = recognizer.recognize(temp_wav, lang_id='eng', timestamp=True, topk=1)
    print(f"[{word_id}] Allosaurus detected sequence:\n{raw_phones.strip()}\n")

    parsed = []
    for line in raw_phones.strip().split("\n"):
        if not line.strip():
            continue
        parts = line.split()
        parsed.append((float(parts[0]), float(parts[1]), parts[2]))

    phonemes_out = []
    for idx, (start_s, dur_s, ipa) in enumerate(parsed):
        ref_file = f"audio/phonemes/{WIKIMEDIA_IPA_URLS[ipa][0]}" if ipa in WIKIMEDIA_IPA_URLS else None

        if ref_file and os.path.exists(ref_file):
            clean_audio_path = ref_file
        else:
            # Fallback to non-overlapping fade-smoothed slice if an unmapped IPA phone appears
            end_s = max(start_s + 0.035, parsed[idx + 1][0] - 0.005) if idx + 1 < len(parsed) else min(len(y) / sr, start_s + max(dur_s, 0.09))
            s_idx = max(0, int((start_s - 0.015) * sr))
            e_idx = min(len(y), int(end_s * sr))
            clean_audio_path = f"audio/slices/{word_id}_p{idx+1}.wav"
            sf.write(clean_audio_path, apply_fade(y[s_idx:e_idx], sr=sr, fade_ms=10), sr)

        phonemes_out.append({
            "id": f"{word_id}_p{idx+1}",
            "kind": classify_phone(ipa),  # "consonant" or "vowel" (zero IPA text sent to UI)
            "audio_url": clean_audio_path
        })

    # --------------------------------------------------------------------------
    # STEP 2: Syllable Duration Segmentation
    # --------------------------------------------------------------------------
    rms = librosa.feature.rms(y=y, frame_length=512, hop_length=160)[0]
    peaks, _ = find_peaks(rms, distance=12, prominence=np.max(rms) * 0.2)
    if len(peaks) == 0:
        peaks = [len(rms) // 2]

    boundaries = [0] + [int((peaks[i] + peaks[i+1]) / 2) for i in range(len(peaks) - 1)] + [len(rms)]
    syllables_out = []
    for idx in range(len(boundaries) - 1):
        s_s = (boundaries[idx] * 160) / sr
        e_s = (boundaries[idx+1] * 160) / sr
        dur = max(0.08, e_s - s_s)
        clip_path = f"audio/slices/{word_id}_s{idx+1}.wav"
        syl_audio = apply_fade(y[int(s_s * sr):int(e_s * sr)], sr=sr, fade_ms=12)
        sf.write(clip_path, syl_audio, sr)

        syllables_out.append({
            "id": f"{word_id}_s{idx+1}",
            "width_px": int(min(180, max(65, dur * 240))),
            "audio_url": clip_path
        })

    # --------------------------------------------------------------------------
    # STEP 3: TorchCREPE F0 Pitch Extraction & Stripped Humming Synthesis
    # --------------------------------------------------------------------------
    audio_tensor = torch.tensor(y).unsqueeze(0)
    pitch, periodicity = torchcrepe.predict(
        audio_tensor,
        sr,
        hop_length=160,
        fmin=65.0,
        fmax=500.0,
        model="full",
        decoder=torchcrepe.decode.viterbi,
        return_periodicity=True,
        device=device
    )
    frequency = pitch.squeeze(0).cpu().numpy()
    conf = periodicity.squeeze(0).cpu().numpy()

    hum_out = f"audio/slices/{word_id}_hum.wav"
    sf.write(hum_out, synthesize_humming_track(y, frequency, conf, sr=sr, hop_length=160), sr)

    tones_out = []
    num_syl = len(syllables_out)
    for idx in range(num_syl):
        s_frame = int((idx / num_syl) * len(frequency))
        e_frame = int(((idx + 1) / num_syl) * len(frequency))
        seg_f0 = frequency[s_frame:e_frame]
        slope = float(seg_f0[-1] - seg_f0[0]) if len(seg_f0) > 1 else 0.0
        style = "solid" if slope > 12 else ("ring" if slope < -12 else "dot")
        tones_out.append({
            "id": f"{word_id}_t{idx+1}",
            "style": style,
            "audio_url": hum_out
        })

    return {
        "id": word_id,
        "visual_url": visual_path,
        "full_audio_url": f"audio/{word_id}.mp3",
        "humming_audio_url": hum_out,
        "phonemes": phonemes_out,
        "syllables": syllables_out,
        "tones": tones_out
    }

# 4. Process all 5 Level 0 vocabulary words
words = [
    ("cat", "/content/cat.mp3", "images/visuals/cat.jpg"),
    ("dog", "/content/dog.mp3", "images/visuals/dog.jpg"),
    ("walk", "/content/walk.mp3", "images/visuals/walk.gif"),
    ("jump", "/content/jump.mp3", "images/visuals/jump.gif"),
    ("tree", "/content/tree.mp3", "images/visuals/tree.jpg")
]

dataset = {"items": [process_word(w, p, v) for w, p, v in words if os.path.exists(p)]}

with open("lesson_data.json", "w") as f:
    json.dump(dataset, f, indent=2)

# 5. Bundle lesson_data.json, audio/slices/, AND audio/phonemes/ into zip
with zipfile.ZipFile("gll_update1_assets.zip", "w") as zf:
    zf.write("lesson_data.json")
    for folder in ["audio/slices", "audio/phonemes"]:
        for root, _, files in os.walk(folder):
            for file in files:
                zf.write(os.path.join(root, file))

print("Done! Download gll_update1_assets.zip and extract into 'Intro to AI Project/'.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
downloading model  latest
from:  https://github.com/xinjli/allosaurus/releases/download/v1.0/latest.tar.gz
to:    /usr/local/lib/python3.13/dist-packages/allosaurus/pretrained
please wait...
Models loaded on device: cpu

Done! Download gll_update1_assets.zip and extract into 'Intro to AI Project/'.
